In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/application_train_clean.csv')

print("Shape:", df.shape)
print("Target rate:", round(df['TARGET'].mean() * 100, 2), "%")

Shape: (307511, 113)
Target rate: 8.07 %


In [2]:
df['DAYS_BIRTH'] = df['DAYS_BIRTH'].abs()
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].abs()

df['AGE_YEARS'] = df['DAYS_BIRTH'] / 365
df['EMPLOYMENT_YEARS'] = df['DAYS_EMPLOYED'] / 365

df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['CREDIT_ANNUITY_RATIO'] = df['AMT_CREDIT'] / df['AMT_ANNUITY']
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / df['AMT_CREDIT']

print("New features created:")
print(df[['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 
          'CREDIT_ANNUITY_RATIO', 'INCOME_PER_PERSON', 
          'GOODS_CREDIT_RATIO']].describe().round(2))

New features created:
       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO  CREDIT_ANNUITY_RATIO  \
count            307511.00             307511.00             307511.00   
mean                  3.96                  0.18                 21.61   
std                   2.69                  0.09                  7.82   
min                   0.00                  0.00                  6.32   
25%                   2.02                  0.11                 15.61   
50%                   3.27                  0.16                 20.00   
75%                   5.16                  0.23                 27.10   
max                  84.74                  1.88                 59.56   

       INCOME_PER_PERSON  GOODS_CREDIT_RATIO  
count          307511.00           307511.00  
mean            93106.34                0.90  
std            101373.31                0.10  
min              2812.50                0.17  
25%             47250.00                0.83  
50%             75000.00     

In [3]:
df['EMPLOYMENT_RATIO'] = df['EMPLOYMENT_YEARS'] / df['AGE_YEARS']

df['AGE_BAND'] = pd.cut(df['AGE_YEARS'], 
                         bins=[0, 25, 35, 45, 55, 100],
                         labels=['<25', '25-35', '35-45', '45-55', '55+'])

df['EXT_SOURCE_MEAN'] = df[['EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
df['EXT_SOURCE_MIN'] = df[['EXT_SOURCE_2', 'EXT_SOURCE_3']].min(axis=1)

df['BUREAU_ENQUIRY_TOTAL'] = (df['AMT_REQ_CREDIT_BUREAU_DAY'] + 
                               df['AMT_REQ_CREDIT_BUREAU_WEEK'] + 
                               df['AMT_REQ_CREDIT_BUREAU_MON'] + 
                               df['AMT_REQ_CREDIT_BUREAU_QRT'])

df['HAS_SOCIAL_CIRCLE_DEFAULT'] = (df['DEF_30_CNT_SOCIAL_CIRCLE'] > 0).astype(int)

age_default = df.groupby('AGE_BAND')['TARGET'].mean().round(4)
print("Default rate by age band:")
print(age_default)

print("\nCorrelation of new features with TARGET:")
new_features = ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 
                'EXT_SOURCE_MEAN', 'EXT_SOURCE_MIN',
                'EMPLOYMENT_RATIO', 'BUREAU_ENQUIRY_TOTAL',
                'HAS_SOCIAL_CIRCLE_DEFAULT']
print(df[new_features + ['TARGET']].corr()['TARGET'].drop('TARGET').sort_values().round(4))



Default rate by age band:
AGE_BAND
<25      0.1230
25-35    0.1067
35-45    0.0840
45-55    0.0706
55+      0.0522
Name: TARGET, dtype: float64

Correlation of new features with TARGET:
EXT_SOURCE_MEAN             -0.2137
EXT_SOURCE_MIN              -0.1899
EMPLOYMENT_RATIO            -0.0445
BUREAU_ENQUIRY_TOTAL        -0.0147
CREDIT_INCOME_RATIO         -0.0077
ANNUITY_INCOME_RATIO         0.0143
HAS_SOCIAL_CIRCLE_DEFAULT    0.0320
Name: TARGET, dtype: float64


In [4]:
def winsorize(df, col, lower=0.01, upper=0.99):
    lo = df[col].quantile(lower)
    hi = df[col].quantile(upper)
    df[col] = df[col].clip(lo, hi)
    return df

cols_to_winsorize = ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',
                     'INCOME_PER_PERSON', 'CREDIT_ANNUITY_RATIO',
                     'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'BUREAU_ENQUIRY_TOTAL']

for col in cols_to_winsorize:
    df = winsorize(df, col)

df = df.drop(columns=['AGE_BAND'])

print("Shape after feature engineering:", df.shape)
print("Any infinites:", np.isinf(df.select_dtypes(include=np.number)).sum().sum())
print("Any nulls:", df.isnull().sum().sum())

df.to_csv('../data/processed/application_train_features.csv', index=False)
print("Saved successfully")

Shape after feature engineering: (307511, 125)
Any infinites: 0
Any nulls: 0
Saved successfully
